In [2]:
import os
import cv2
import matplotlib.pyplot as plt


def visualize_yolo_dataset(
    dataset_dir,
    split="train",
    n=5,
    class_names=None,
    figsize=(6, 6)
):
    """
    Visualize first n images from a YOLO dataset with bounding boxes.

    Args:
        dataset_dir (str): Root YOLO dataset directory
                           (contains images/ and labels/)
        split (str): "train", "val", or "test"
        n (int): Number of images to display
        class_names (list or dict): Optional class names
        figsize (tuple): Figure size for matplotlib
    """

    images_dir = os.path.join(dataset_dir, "images", split)
    labels_dir = os.path.join(dataset_dir, "labels", split)

    image_files = sorted([
        f for f in os.listdir(images_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

    if len(image_files) == 0:
        raise RuntimeError("No images found")

    image_files = image_files[:n]

    for img_name in image_files:
        img_path = os.path.join(images_dir, img_name)
        label_path = os.path.join(
            labels_dir,
            os.path.splitext(img_name)[0] + ".txt"
        )

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        # ---- Read labels ----
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()
        else:
            lines = []

        # ---- Draw boxes ----
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id, xc, yc, w, h = map(float, parts)
            class_id = int(class_id)

            # Convert YOLO → pixel coords
            x1 = int((xc - w / 2) * W)
            y1 = int((yc - h / 2) * H)
            x2 = int((xc + w / 2) * W)
            y2 = int((yc + h / 2) * H)

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

            if class_names is not None:
                label = (
                    class_names[class_id]
                    if isinstance(class_names, (list, dict))
                    else str(class_id)
                )
                cv2.putText(
                    img,
                    label,
                    (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 0, 0),
                    1,
                    cv2.LINE_AA
                )

        # ---- Display ----
        plt.figure(figsize=figsize)
        plt.imshow(img)
        plt.title(img_name)
        plt.axis("off")
        plt.show()


In [ ]:
visualize_yolo_dataset(
    dataset_dir="C:/Users/chris/Desktop/YOLO_dataset",
    split="train",
    n=5
)


In [7]:
import random
import shutil
from pathlib import Path


def split_yolo_dataset(
    dataset_dir,
    train_ratio=0.7,
    val_ratio=0.1,
    test_ratio=0.2,
    seed=42,
    image_exts=(".png", ".jpg", ".jpeg"),
    move_files=True,  # True = move val/test, False = copy
):
    """
    Split YOLO dataset by MOVING validation/test samples out of images/train.

    Initial structure:
        images/train/
        labels/train/

    Final structure:
        images/{train,val,test}/
        labels/{train,val,test}/
    """

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    dataset_dir = Path(dataset_dir)

    img_train = dataset_dir / "images" / "train"
    lbl_train = dataset_dir / "labels" / "train"

    assert img_train.exists(), "images/train not found"
    assert lbl_train.exists(), "labels/train not found"

    # Collect image–label pairs
    pairs = []
    for img in img_train.iterdir():
        if img.suffix.lower() not in image_exts:
            continue
        lbl = lbl_train / (img.stem + ".txt")
        if lbl.exists():
            pairs.append((img, lbl))

    print(f"Found {len(pairs)} image–label pairs")

    # Shuffle deterministically
    random.seed(seed)
    random.shuffle(pairs)

    n_total = len(pairs)
    n_train = int(train_ratio * n_total)
    n_val   = int(val_ratio * n_total)

    train_pairs = pairs[:n_train]
    val_pairs   = pairs[n_train:n_train + n_val]
    test_pairs  = pairs[n_train + n_val:]

    # Create folders
    for split in ["val", "test"]:
        (dataset_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (dataset_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    def transfer(pairs, split):
        for img, lbl in pairs:
            dst_img = dataset_dir / "images" / split / img.name
            dst_lbl = dataset_dir / "labels" / split / lbl.name

            if move_files:
                shutil.move(img, dst_img)
                shutil.move(lbl, dst_lbl)
            else:
                shutil.copy2(img, dst_img)
                shutil.copy2(lbl, dst_lbl)

    # Move only val & test
    transfer(val_pairs, "val")
    transfer(test_pairs, "test")

    print(f"train: {len(train_pairs)} (left in place)")
    print(f"val:   {len(val_pairs)}")
    print(f"test:  {len(test_pairs)}")
    print("Dataset split complete")


In [8]:
split_yolo_dataset(
    dataset_dir="C:/Users/chris/Desktop/YOLO_dataset",
    train_ratio=0.7,
    val_ratio=0.1,
    test_ratio=0.2,
    seed=123
)


Found 1009 image–label pairs
train: 706 (left in place)
val:   100
test:  203
Dataset split complete
